# Lecture 07 — Functions, cont.
### EES 3350/5350 · Python in Earth Science · Friday, September 11, 2026

**Learning objectives for today**
- unpack multiple return values from a function into separate named variables
- import and use functions from Python's `statistics` module, and explore on your own
- call the same function using data stored in different structures (nested dict. vs. parallel lists)
- watch a function's behavior branch depending on its input (a first look at branching...)
-----

## Part 0 : picking up where we left off
Let's recreate `gully_data`

In [1]:
import math

gully_data = {
    "Heil Ranch Gully 1": {"normal_infiltration": 16, "postfire_infiltration": 2.25, "drainage_area": 0.1538},
    "Heil Ranch Gully 2": {"normal_infiltration": 16, "postfire_infiltration": 11.77, "drainage_area": 0.2523},
    "Geer Canyon":  {"normal_infiltration": 22, "postfire_infiltration": 3.5, "drainage_area": 7.325},
    "Central Gulch":  {"normal_infiltration": 16, "postfire_infiltration": 4.0, "drainage_area": 0.1515},
}

---
## Part 1 : catching multiple returns
On Wednesday, we wrote `check_debris_flow()`, which returns *two* values as a tuple, but we never actually called it! Let's fix that.

In [2]:
def check_debris_flow(I, D):
    """
    Determine whether a storm is predicted to trigger a debris flow using the 
    Caine (1980) intensity-duration threshold.

    Parameters
    ----------
    I : float
        storm rainfall intensity (mm/hr)
    D : float
        storm duration (hr)

    Returns
    ---------
    threshold : float
        Cain (1980) threshold intensity (mm/hr)
    debris_flow : bool
        whether a debris flow is predicted (I >= threshold)
    """
    threshold = 14.82 * D**(-0.39)
    debris_flow = I >= threshold
    
    return threshold, debris_flow

Let's use this one to look at a storm from September 2013:
- 12.5 hours long
- peak intensity around 10 mm/hr

<div class="alert alert-info">

### Exercise 1

Use `check_debris_flow()` to see if the September 2013 flood reaches the Cain (1980) threshold for debris flows. **Tell me what to type.**
    
</div>

In [ ]:
# your answer here
result = check_debris_flow(________________)
print(result)
print(type(result))

This is a **tuple**. You can index into it (`result[0]`, `result[1]`), or you can unpack it directly into two named variables in one line: 

In [ ]:
storm_threshold, storm_debris_flow = check_debris_flow(________)
print(storm_threshold)
print(storm_debris_flow)

<div class="alert alert-success">

**Tip**: Unpacking is just a convenience. `result[0]` and `result[1]` get you the exact same two values. But giving each one its own descriptive name makes the rest of your code much easier to read, especially once you're juggling outputs from several functions at once.
    
</div>

<div class="alert alert-info">

### Exercise 2
Check the same storm duration (12.5), but with a lower peak intensity of 4mm/hr. Unpack the results into two variables called `low_threshold` and `low_debris_flow`, then print a sentence stating whether a debris flow is predicted.
    
</div>

In [ ]:
# your answer here

----
## Part 2 : Using someone else's module -- `statistics`
Just like `math` back in Week 2, Python's standard library includes a `statistics` module full of ready-made functions for summarizing numeric data. Let's import it the same way we imported `math`, but with a shortened alias.

In [4]:
import statistics as stats

Let's pull the postfire infiltration rate for all four gullies into a single list, then use `stats.mean()` to summarize it:

In [5]:
postfire_rates = [
    gully_data["Heil Ranch Gully 1"]["postfire_infiltration"],
    gully_data["Heil Ranch Gully 2"]["postfire_infiltration"],
    gully_data["Geer Canyon"]["postfire_infiltration"],
    gully_data["Central Gulch"]["postfire_infiltration"]
]

print(postfire_rates)

[2.25, 11.77, 3.5, 4.0]


In [6]:
print(stats.mean(postfire_rates))

5.38


<div class="alert alert-info">

### Exercise 3
Use `help(stats)` (or `dir(stats)`) to browse what else lives in the `statistics` module. Find **two** other functions you haven't used yet, run each of them on `postfire_rates`, and add a markdown cell noting what each one calculates.
    
</div>

In [7]:
# your answer here

---
## Part 3 : Same function, different data structure
Back in Exercise 6 of Lecture 06, you wrote `total_discharge()` using `gully_data`, a dictionary of dictionaries. Let's rebuild it here.

In [8]:
def total_discharge(P, ET, S, area_km2, duration):
    """
    Calculate a storm's total runoff volume at a gully's outlet.

    Parameters
    ----------
    P : float
        precipitation rate (mm/hr)
    ET : float
        evapotranspiration rate (mm/hr)
    S : float
        infiltration rate (mm/hr)
    area_km2 : float
        gully drainage area (km^2)
    duration : float
        storm duration (hr)

    Returns
    -------
    float
        total runoff volume (km^3)
    """
    Q_rate = P - ET - S
    Q_rate_km = Q_rate / (1000 * 1000)
    discharge = Q_rate_km * area_km2
    runoff_volume = discharge * duration
    return runoff_volume

print(total_discharge(30, 3, gully_data["Central Gulch"]["postfire_infiltration"], gully_data["Central Gulch"]["drainage_area"], 2))

6.969e-06


That worked because `gully_data` is a dictionary, and we pulled each value out by key. But `total_discharge()` itself doesn't know or care that the data came from a dictionary. It just wants four n umbers and a duration. Let's prove that by storing the exact same data as **parallel lists** instead.

In [9]:
gully_names    = ["Heil Ranch Gully 1", "Heil Ranch Gully 2", "Geer Canyon", "Central Gulch"]
postfire_infil = [2.25, 11.77, 3.5, 4.0]
areas          = [0.1538, 0.2523, 7.325, 0.1515]

Now, instead of looking a gully up by name (a key), we look it up by its **position** in each list. All three lists are in the same order, so index `n` always refers to the same gully across all of them.

In [98]:
n = 3 # Central Gulch is at index 3

print(gully_names[n])
print(total_discharge(30, 3, postfire_infil[n], areas[n],2))

Central Gulch
6.969e-06


<div class="alert alert-success">

**Tip**: Compare this cell's output to the one from the function definition cell above. It's the same function, the same math, the same answer. Only the way we pulled numbers out changed. This is a really common pattern in scientific programming: keep your functions general, and let the data structure be whatever's most convenient for the situation.
    
</div>

<div class="alert alert-info">

### Exercise 4
Using the parallel lists above (not `gully_data`), calculate `total_discharge()` for **Heil Ranch Gully 1**, using the same storm as before:
- 30 mm/hr precipitation
- 3 mm/hr evapotranspiration
- 2 hr duration
    
</div>

In [11]:
# your answer here

##### Let's write a different version of this function to take the tuples `location_A` and `location_B` which will be `[lat, long]` for each location.

In [97]:
def degrees_to_kilometers(location_A, location_B):
    """
    Convert an east-west distance in degrees longitude to kilometers,
    accounting for the convergence of meridians toward the poles.
    
    Parameters
    ----------
    location_A : tuple
        (latitude, longitude) in decimal degrees
    location_B : tuple
        (latitude, longitude) in decimal degrees
        
    Returns
    -------
    float
        distance in kilometers
    """
    deg_dist = abs(location_B[1] - location_A[1])
    conversion = 111 * math.cos(math.radians(location_A[0]))
    km_dist = deg_dist * conversion
    return km_dist

We can then test it on two points:

In [96]:
pointA = (40.0, -80.0)
pointB = (40.0, -75.0)

print(degrees_to_kilometers(pointA, pointB))

425.15466593103275


### But what about **north-south** distance?
`degrees_to_kilometers()` right now handles east-west distance (same latitude, different longitude). But what if two points differ in *latitude* instead? You might guess the same "111 km per degree" constant applies. 

##### On a perfectly round sphere, it would! But Earth isn't a perfect sphere...it's an **oblate spheroid**, very slightly flattened at the poles:

In [20]:
from IPython.display import Video

Video("../figures/geoid_GOCO06s.mp4", width=600)
#Video("../../figures/geoid_GOCO06s.mp4", width=600)

<span style="font-size:0.85em;color:#6b7a8d;"><em> Kvas, A., Brockmann, J. M., Krauss, S., Schubert, T., Gruber, T., Meyer, U., Mayer-Gürr, T., Schuh, W.-D., Jäggi, A., and Pail, R.: GOCO06s – a satellite-only global gravity field model, Earth Syst. Sci. Data, 13, 99–118, https://doi.org/10.5194/essd-13-99-2021, 2021. </em></span>

Because of this, the length of one degree of latitude itself stretches very slightly as you move from the equator toward the poles:
- near the equator: about 110.6 km per degree of latitude
- near the poles: about 111.7 km per degree of latitude

These values come from real WGS84 ellipsoid figures (roughly a 1% difference pole-to-equator).

##### This is a separate, much smaller effect than the `cos(latitude)` shrinkage above, and it only applies to north-south distances.

#### Let's write a function that picks the right constant depending on latitude band and then can also calculate that distance. Watch what it does first! (we'll learn the formal syntax behind `if`/`elif`/`else` soon), but you already have everything else you need to read this.

In [75]:
def north_south_distance(location_A, location_B):
    """
    Lookup the local "km per degree" conversion factor and use it to calculate the north-south distance between two locations.
    The conversion factor varies slightly with latitude because Earth is an oblate spheroid, not a perfect sphere.

    Parameters
    ----------
    location_A : tuple
        (latitude, longitude) in decimal degrees
    location_B : tuple
        (latitude, longitude) in decimal degrees

    Returns
    -------
    conversion : float
        approximate km per degree of latitude at location_A's location
    distance : float
        north-south distance in kilometers
    """
    if abs(location_A[0]) < 30:
        conversion = 110.6
    elif abs(location_A[0]) < 60:
        conversion = 111.0
    else:
        conversion = 111.5

    deg_dist = abs(location_B[0] - location_A[0])
    distance = deg_dist * conversion
    
    return conversion, distance

Let's test it on two points

In [76]:
pointC = (10.0, -80.0)
pointD = (40.0, -80.0)

conversion, distance = north_south_distance(pointC, pointD)

print(f"The conversion for these points is {conversion} km.")
print(f"Which means that these two points are {distance} km apart.")

The conversion for these points is 110.6 km.
Which means that these two points are 3318.0 km apart.


<div class="alert alert-success">

**Tip:** Look closely at `pointC` and `pointD` above — they're at 10° and 40° latitude, which means the true straight-line path between them actually *crosses* the 30° band boundary. `north_south_distance()` doesn't know that — it looks up **one** conversion factor based on `location_A`'s latitude (110.6, since 10° is in the 0°–30° band) and applies it to the *entire* 30-degree span, even the part that's really in the 30°–60° band.

A more careful version would split the trip at the boundary and add the pieces separately: 20 degrees at 110.6 km/degree, plus 10 degrees at 111.0 km/degree, giving 3322 km instead of the 3318 km we got. That's a small difference here (about 0.1%), but it's a real, disclosed simplification, not a hidden one -- and it's a limitation worth remembering. Once we have `for` loops, this becomes easy to fix properly: instead of one conversion factor for the whole trip, you can step through the distance in small increments and use whichever band applies at each step.

</div>

#### Now that we have this function that will give us the north-south distance, let's go back into our `degrees_to_kilometers()` and edit it to use one of the outputs of the new `north_south_distance()` function instead of just 111. We'll call this reworked function `east_west_distance()`.

In [77]:
def east_west_distance(location_A, location_B):
    """
    Convert an east-west distance in degrees longitude to kilometers,
    accounting for the convergence of meridians toward the poles and for
    the small latitude-dependent variation in Earth's curvature.
    
    Parameters
    ----------
    location_A : tuple
        (latitude, longitude) in decimal degrees
    location_B : tuple
        (latitude, longitude) in decimal degrees
        
    Returns
    -------
    float
        distance in kilometers
    """
    deg_dist = abs(location_B[1] - location_A[1])
    conversion, ns_distance = north_south_distance(location_A, location_B)
    full_conversion = conversion * math.cos(math.radians(location_A[0])) # changed 111 to a use of the `north_south_distance()` function
    km_dist = deg_dist * full_conversion
    return km_dist

Test it on those same two points 5 degrees apart in longitude:

In [79]:
dist_ew = east_west_distance(pointA, pointB)

print(f"These two points are {dist_ew:.2f} km apart.")

These two points are 425.15 km apart.


### Let's see all of that in action.
Two points that are both 5 degrees apart in longitude, but at different latitudes, will be very different distances apart:

In [80]:
pointE = (25.0, -80.0)
pointF = (25.0, -75.0)

lat_25 = east_west_distance(pointE, pointF)
print(f"5 degrees longitude at 25 degrees latitude is: {lat_25:.2f} km")

5 degrees longitude at 25 degrees latitude is: 501.19 km


In [81]:
pointG = (70.0, -80.0)
pointH = (70.0, -75.0)

lat_70 = east_west_distance(pointG, pointH)
print(f"5 degrees longitude at 70 degrees latitude is: {lat_70:.2f}")

5 degrees longitude at 70 degrees latitude is: 190.68


### So what actually matters more here?
501 km vs. 191 km is a huge difference, but is that because of the new north-south band correction? or something else? Let's check by bringing back the *original* version of this function `degrees_to_kilometers()`, which used one flat constant (111) everywhere and compare it to `east_west_distance()` at the exact same two latitudes:

In [86]:
print("At 25 degrees:")
print("  new:", round(east_west_distance(pointE, pointF), 2), "km")
print("  old:", round(degrees_to_kilometers(pointE, pointF), 2), "km")
print()
print("At 70 degrees:")
print("  new:", round(east_west_distance(pointG, pointH), 2), "km")
print("  old:", round(degrees_to_kilometers(pointG, pointH), 2), "km")

At 25 degrees:
  new: 501.19 km
  old: 503.0 km

At 70 degrees:
  new: 190.68 km
  old: 189.82 km


<div class="alert alert-info">

### Exercise 5

How different are the values at 25 degrees latitude? at 70 degrees latitude? Give your answer as a **percent difference**, using anything you've learned so far. 
    
</div>

In [ ]:
# your answer here

<div class="alert alert-block alert-warning">

### Question 1

Of the two corrections you've explored in this section -- the `cos(latitude)` adjustment (which accounts for how much closer together lines of longitude get near the poles) and the latitude-band adjustment (which accounts for Earth's oblate shape) -- which one has a bigger effect on your final distance calculation? Use your results from above to support your answer.   

</div>

> your answer here

### Putting it all together
`east_west_distance()` and `north_south_distance()` each only handle one direction, but real places are almost never due east, due west, due north, or due south of each other. They usually differ in both latitude and longitude at once. Let's combine both functions into one that handles the general case, treating the east-west and north-south components as the two legs of a right triangle and combining them with the Pythagorean theorem.

In [103]:
def total_distance(location_A, location_B):
    """
    Approximate the straight-line distance between two locations that 
    differ in both latitude and longitude, by treating the east-west and 
    north-south components as legs of a right triangle.

    Parameters
    ----------
    location_A : tuple
        (latitude, longitude) in decimal degrees
    location_B : tuple
        (latitude, longitude) in decimal degrees

    Returns
    -------
    float
        approximate distance in kilometers
    """
    ew_distance = east_west_distance(location_A, location_B)
    conversion, ns_distance = north_south_distance(location_A, location_B)
    total_dist = math.sqrt((ew_distance**2) + (ns_distance**2))

    return total_dist

Let's test it on two real cities:

In [107]:
nashville = (36.16, -86.78)
denver = (39.74, -104.99)

print(round(total_distance(nashville, denver), 2))

1679.63


#### That last one prints 1679.63 — remarkably close to Nashville-Denver's real-world distance (~1680 km), a nice sanity check to point out live.

----
## Extra Practice
Here are three more problems pulling together everything from today: tuple unpacking, the statistics module, and mapping data structures onto functions.

<div class="alert alert-info">

### Exercise 6
Write a function called `discharge_report()` that takes a gully's index `n` and:
- calculates `total_discharge()` for that gully using the same storm (30 mm/hr precip, 3 mm/hr ET, 2 hr duration)
- outputs the volume
- prints a sentence with the gully's name and its total runoff volume
- includes a docstring (feel free to copy and paste the formatting)

Test it on Geer Canyon.
    
</div>

In [89]:
# your answer here

<div class="alert alert-info">

### Exercise 7
Use `discharge_report()` (or `total_discharge()`) directly to :
- calculate the total runoff volume for **all four gullies**
- save each one to its own variable
- use `stats.mean()` to find the average total discharge across all four gullies
    
</div>

In [91]:
# your answer here

<div class="alert alert-info">

### Exercise 8
Which city is closer to Nashville, TN `(36.16, -86.78)`: Denver, CO `(39.74, -104.99)`, or Chicago, IL `(41.88, -87.63)`? Use `total_distance()` to calculate both distances, print them, and then state in a markdown cell which city is closer and by roughly how much.    
</div>

In [92]:
# your answer here 

> your explanation here